# No-show predictor
Predict the share of booked passengers who will not show up for a flight.

**Flow:** data -> explore -> split -> train -> evaluate vs baseline -> explain -> predict -> save.

The data is synthetic (generated below), so results show the method, not real airline performance.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.inspection import permutation_importance

## 2. Dataset
We generate 5,000 flights. Each flight has a hidden rule for its no-show rate (flexible tickets, connecting passengers, early flights and Mondays/Fridays raise it) plus random noise. The model must learn this rule from the data.

*Optional:* to use the provided `flights.csv` instead, upload it in the Colab file panel and run the commented line.

In [ ]:
ROUTES = ["AMS-LHR", "AMS-JFK", "AMS-CDG", "AMS-MAD", "AMS-DXB", "AMS-BCN", "AMS-ATH", "AMS-OSL"]

def generate_flights(n=5000, seed=42):
    rng = np.random.default_rng(seed)
    hour = rng.integers(5, 23, n)
    dow = rng.integers(0, 7, n)
    advance = rng.gamma(4.0, 12.0, n).round(1)
    flex = rng.beta(2, 5, n).round(3)
    connecting = rng.beta(2, 6, n).round(3)
    rate = (0.02 + 0.06 * flex + 0.04 * connecting + 0.0003 * advance
            + np.where(hour < 7, 0.008, 0.0)
            + np.where(np.isin(dow, [0, 4]), 0.006, 0.0)
            + rng.normal(0, 0.01, n))
    return pd.DataFrame({
        "route": rng.choice(ROUTES, n),
        "hour": hour,
        "day_of_week": dow,
        "month": rng.integers(1, 13, n),
        "booked_passengers": rng.integers(60, 300, n),
        "advance_booking_days": advance,
        "flex_share": flex,
        "connecting_share": connecting,
        "no_show_rate": np.clip(rate, 0.005, 0.25),
    })

df = generate_flights()
# df = pd.read_csv("flights.csv")   # use this instead if you uploaded the CSV
df.to_csv("flights.csv", index=False)
df.head()

## 3. Explore the data

In [ ]:
print(df.shape)
print(df.isna().sum().sum(), "missing values")
df.describe().round(3)

In [ ]:
num = df.select_dtypes("number").drop(columns="no_show_rate")
corr = num.corrwith(df["no_show_rate"]).sort_values()
corr.plot.barh(title="Correlation with no_show_rate")
plt.tight_layout()
plt.show()

## 4. Train / test split
The model trains on 80% of flights and is judged on the 20% it has never seen.

In [ ]:
CATEGORICAL = ["route"]
NUMERIC = ["hour", "day_of_week", "month", "booked_passengers",
           "advance_booking_days", "flex_share", "connecting_share"]
FEATURES = CATEGORICAL + NUMERIC
TARGET = "no_show_rate"

train, test = train_test_split(df, test_size=0.2, random_state=42)
len(train), len(test)

## 5. Model
A pipeline keeps preprocessing and model together, so the same steps are applied in training and in production. Route is one-hot encoded; unknown routes are ignored instead of crashing.

In [ ]:
class NoShowModel:
    def __init__(self, random_state=42):
        prep = ColumnTransformer(
            [("route", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)],
            remainder="passthrough")
        self.pipeline = Pipeline([
            ("prep", prep),
            ("reg", GradientBoostingRegressor(random_state=random_state)),
        ])

    def fit(self, df):
        self.pipeline.fit(df[FEATURES], df[TARGET])
        return self

    def predict(self, df):
        return np.clip(self.pipeline.predict(df[FEATURES]), 0.0, 1.0)

    def evaluate(self, df):
        baseline = np.full(len(df), df[TARGET].mean())
        return {"mae": mean_absolute_error(df[TARGET], self.predict(df)),
                "baseline_mae": mean_absolute_error(df[TARGET], baseline)}

model = NoShowModel().fit(train)

## 6. Evaluate
MAE is the average gap between predicted and actual no-show rate. The baseline always predicts the average rate. A useful model must beat it.

In [ ]:
metrics = model.evaluate(test)
print(f"Model MAE:    {metrics['mae']:.4f}")
print(f"Baseline MAE: {metrics['baseline_mae']:.4f}")
print(f"Improvement:  {1 - metrics['mae'] / metrics['baseline_mae']:.0%}")

In [ ]:
pred = model.predict(test)
plt.scatter(test[TARGET], pred, alpha=0.3, s=8)
lim = [0, test[TARGET].max()]
plt.plot(lim, lim, "r--")
plt.xlabel("Actual no-show rate")
plt.ylabel("Predicted no-show rate")
plt.title("Predicted vs actual (closer to the line is better)")
plt.show()

## 7. What drives the predictions?
Permutation importance shuffles one feature at a time and measures how much the error rises. Bigger drop = more important.

In [ ]:
imp = permutation_importance(model.pipeline, test[FEATURES], test[TARGET],
                             n_repeats=5, random_state=42,
                             scoring="neg_mean_absolute_error")
pd.Series(imp.importances_mean, index=FEATURES).sort_values().plot.barh(
    title="Permutation importance")
plt.tight_layout()
plt.show()

## 8. Predict for one new flight

In [ ]:
flight = pd.DataFrame([{
    "route": "AMS-LHR", "hour": 8, "day_of_week": 4, "month": 6,
    "booked_passengers": 180, "advance_booking_days": 40,
    "flex_share": 0.3, "connecting_share": 0.2}])
rate = model.predict(flight)[0]
print(f"Predicted no-show rate: {rate:.1%}")
print(f"Expected no-shows: {rate * 180:.1f} of 180 booked")

## 9. Save the model
This is the artifact an API would load in production.

In [ ]:
joblib.dump(model, "model.joblib")
# from google.colab import files
# files.download("model.joblib")